In [1]:
import mlflow
import pandas as pd


/Users/siddharthaganguli/Desktop/Yt-sentiment/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import dagshub
dagshub.init(repo_owner="siddharthaganguli0093", repo_name="youtube_Comment_analyzer", mlflow=True)


Accessing as siddharthaganguli0093

Initialized MLflow to track repo "siddharthaganguli0093/youtube_Comment_analyzer"

Repository siddharthaganguli0093/youtube_Comment_analyzer initialized!

In [3]:
mlflow.set_tracking_uri("https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow")


In [4]:
df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
df.head()


,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [5]:
df.dropna(inplace=True)


In [6]:
df.drop_duplicates(inplace=True)


In [7]:
df = df[(df['clean_comment'].str.strip() != '')]


In [8]:
df[(df['clean_comment'].str.strip() == '')]


,clean_comment,category


In [9]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [10]:
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/siddharthaganguli/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/siddharthaganguli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [11]:
# Define the preprocessing function
def preprocess_comment(comment):
    # Convert to lowercase
    comment = comment.lower()

    # Remove trailing and leading whitespaces
    comment = comment.strip()

    # Remove newline characters
    comment = re.sub(r'\n', ' ', comment)

    # Remove non-alphanumeric characters, except punctuation
    comment = re.sub(r'[^A-Za-z0-9\s!?.,]', '', comment)

    # Remove stopwords but retain important ones for sentiment analysis
    stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
    comment = ' '.join([word for word in comment.split() if word not in stop_words])

    # Lemmatize the words
    lemmatizer = WordNetLemmatizer()
    comment = ' '.join([lemmatizer.lemmatize(word) for word in comment.split()])

    return comment


In [12]:
df['clean_comment'] = df['clean_comment'].apply(preprocess_comment)


In [13]:
df.head()


,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [14]:
import mlflow
print(mlflow.__version__)

import mlflow.sklearn


3.14.0


In [15]:
import mlflow
from mlflow.types.schema import DataType
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [16]:
vectorizer = CountVectorizer(max_features=10000)


In [17]:
X = vectorizer.fit_transform(df['clean_comment']).toarray()
y = df['category']  # Assuming 'sentiment' is the target variable (0 or 1 for binary classification)


In [18]:
X


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(36793, 10000))

In [19]:
X.shape


(36793, 10000)

In [20]:
mlflow.set_experiment("RF Baseline")


<Experiment: artifact_location='mlflow-artifacts:/d99b104e94f946ab8397ab0982204e48', creation_time=1783889812070, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783889812070, lifecycle_stage='active', name='RF Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [24]:
import os
from pathlib import Path
import joblib
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================
# Create artifacts directory
# =====================================================
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

# =====================================================
# Train-Test Split
# =====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =====================================================
# MLflow Run
# =====================================================
with mlflow.start_run(run_name="RandomForest_Baseline_TrainTestSplit") as run:

    print("Run ID:", run.info.run_id)

    # -----------------------------
    # Tags
    # -----------------------------
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("model_type", "RandomForestClassifier")
    mlflow.set_tag(
        "description",
        "Random Forest baseline using Bag of Words features"
    )

    # -----------------------------
    # Parameters
    # -----------------------------
    n_estimators = 150
    max_depth = 15

    mlflow.log_param("vectorizer_type", "CountVectorizer")
    mlflow.log_param("vectorizer_max_features", vectorizer.max_features)

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    # -----------------------------
    # Model
    # -----------------------------
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    # -----------------------------
    # Predictions
    # -----------------------------
    y_pred = model.predict(X_test)

    # -----------------------------
    # Metrics
    # -----------------------------
    accuracy = accuracy_score(y_test, y_pred)

    mlflow.log_metric("accuracy", accuracy)

    report = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    for label, values in report.items():
        if isinstance(values, dict):
            for metric, score in values.items():
                mlflow.log_metric(f"{label}_{metric}", score)

    # -----------------------------
    # Confusion Matrix
    # -----------------------------
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")

    cm_path = artifact_dir / "confusion_matrix.png"

    plt.savefig(cm_path, dpi=300, bbox_inches="tight")
    plt.close()

    mlflow.log_artifact(str(cm_path))

    # -----------------------------
    # Save Dataset
    # -----------------------------
    dataset_path = artifact_dir / "dataset.csv"

    df.to_csv(dataset_path, index=False)

    mlflow.log_artifact(str(dataset_path))

    # -----------------------------
    # Save Model Locally
    # -----------------------------
    model_path = artifact_dir / "random_forest.pkl"

    joblib.dump(model, model_path)

    mlflow.log_artifact(str(model_path))

    # -----------------------------
    # Save Vectorizer
    # -----------------------------
    vectorizer_path = artifact_dir / "count_vectorizer.pkl"

    joblib.dump(vectorizer, vectorizer_path)

    mlflow.log_artifact(str(vectorizer_path))

    # -----------------------------
    # MLflow Model
    # -----------------------------
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="random_forest_model"
    )

    print(model_info)

    print("Artifact URI:", mlflow.get_artifact_uri())

print(f"\nAccuracy : {accuracy:.4f}")
print("Training Completed Successfully!")


Run ID: 997c7a4039854339a72c6d627ed1a8fd
Artifact URI: mlflow-artifacts:/d99b104e94f946ab8397ab0982204e48/997c7a4039854339a72c6d627ed1a8fd/artifacts
🏃 View run RandomForest_Baseline_TrainTestSplit at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/1/runs/997c7a4039854339a72c6d627ed1a8fd
🧪 View experiment at: https://dagshub.com/siddharthaganguli0093/youtube_Comment_analyzer.mlflow/#/experiments/1

Accuracy : 0.6479
Training Completed Successfully!


In [ ]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

          -1       1.00      0.01      0.01      1650
           0       0.66      0.83      0.74      2555
           1       0.63      0.84      0.72      3154

    accuracy                           0.65      7359
   macro avg       0.77      0.56      0.49      7359
weighted avg       0.73      0.65      0.57      7359

